<a href="https://colab.research.google.com/github/OdysseusPolymetis/enssib_class/blob/main/NER_for_network_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Reconnaissance d'entités nommées en français**
---

Une entité nommée est une expression linguistique référentielle : pour simplifier, il s'agit d'entités culturellement reconnaissables et référençables. Exemple : Rome est une `LOC`, César est un `PER`, etc.

Il existe de très nombreux modèles pour les langues modernes, c'est beaucoup moins fréquent pour les langues anciennes. Les premiers modèles NER sur transformers ont été ciblés par langue (exemple, le modèle d'Ugarit), mais de plus en plus on s'oriente vers des modèles massifs multilingues, qui comprennent aussi le grec ancien (Roberta large comprend maintenant le grec ancien, plutôt classique).

In [ ]:
!pip install spacy

In [ ]:
!python -m spacy download fr_core_news_sm

## **1. Les étiquettes des NER**

Petite explication des étiquettes (classiques, mais il peut y en avoir plus) :
*   **O** : Outside, pas d'entité reconnaissable
*   **S-PER** : Personnage/Personne, Single, l'entité est faite d'un seul mot
*   **B-PER** : Personnage/Personne, Begin, le mot est le début d'une entité
*   **E-PER** : Personnage/Personne, End, dernier mot de l'entité
*   **I-PER** : Personnage/Personne, Inside, mot au milieu de l'entité
*   **S-MISC** : Miscalleneous (ni `PER` ni `LOC`), Single, mot seul
*   **B-MISC** : Miscalleneous (ni `PER` ni `LOC`), Begin, début de l'entité
*   **E-MISC** : Miscalleneous (ni `PER` ni `LOC`), End, dernier mot de l'entité
*   **I-MISC** : Miscalleneous (ni `PER` ni `LOC`), Inside, mot au milieu de l'entité
*   **S-LOC** : Location, Single, l'entité est faite d'un seul mot
*   **B-LOC** : Location, Begin, début de l'entité
*   **E-LOC** : Location, End, fin de l'entité
*   **I-LOC** : Location, Inside, mot au milieu de l'entité
*   **\<START>** : marqueur crf pour le modèle (début)
*   **\<STOP>** : marqueur crf pour le modèle (fin)








## **2. Long texte et représentation graphique**

**⚠** Je le précise tout de suite, le résultat ne peut pas être parfait. Il y aura forcément des erreurs, d'autant que les données annotées sont limitées.

Nous allons reprendre l'_Odyssée_, en la téléchargeant dans la cellule suivante. Mais vous pouvez mettre votre propre texte (latin ou grec, nous allons utiliser le gros modèle, et ça prendra du temps).

In [ ]:
import re
import csv
import hashlib
import unicodedata as ud
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
import pandas as pd
import spacy
from google.colab import files

In [ ]:
KEEP_LABELS = {"PER"}
COOC_WINDOW = "sentence"
WINDOW_SIZE_TOKENS = 30
MIN_ENTITY_COUNT = 2
MIN_EDGE_WEIGHT = 1
KEEP_DIACRITICS = False

_WS = re.compile(r"\s+", re.UNICODE)

OUTDIR = Path("/content/gephi_out")
OUTDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
nlp = spacy.load(
    "fr_core_news_sm",
    disable=["tagger", "lemmatizer", "attribute_ruler"]
)

In [ ]:
nlp.max_length = 10_000_000

In [ ]:
uploaded = files.upload()
texts = []
for fname, data in uploaded.items():
    try:
        txt = data.decode("utf-8")
    except UnicodeDecodeError:
        txt = data.decode("latin-1", errors="ignore")
    texts.append((Path(fname).stem, txt))

len(texts), [n for n,_ in texts][:5]

In [ ]:
def strip_diacritics(s: str) -> str:
    s = ud.normalize("NFD", s)
    s = "".join(ch for ch in s if ud.category(ch) != "Mn")
    return ud.normalize("NFC", s)

In [ ]:
def normalize_key(label: str, keep_diacritics: bool = False) -> str:
    s = label.strip()
    s = _WS.sub(" ", s)
    s = s.strip(" \t\n\r\"'«»“”‘’.,;:!?()[]{}")
    s = s.lower()
    if not keep_diacritics:
        s = strip_diacritics(s)
    return s

In [ ]:
def make_id(key: str, ent_type: str) -> str:
    h = hashlib.sha1(f"{ent_type}::{key}".encode("utf-8")).hexdigest()[:12]
    return f"n_{h}"

In [ ]:
def extract_entities(doc):
    ents = []
    sent_index_by_token = {}
    for si, sent in enumerate(doc.sents):
        for t in sent:
            sent_index_by_token[t.i] = si

    for ent in doc.ents:
        if KEEP_LABELS and ent.label_ not in KEEP_LABELS:
            continue
        key = normalize_key(ent.text, keep_diacritics=KEEP_DIACRITICS)
        if not key:
            continue
        ents.append({
            "text": ent.text,
            "label": ent.label_,
            "key": key,
            "sent_i": sent_index_by_token.get(ent.start, 0),
            "tok_i": int(ent.start),
        })
    return ents

In [ ]:
def cooccurrence_sentence(ents):
    by_sent = defaultdict(list)
    for e in ents:
        by_sent[e["sent_i"]].append(e)

    edges = Counter()
    for _, elist in by_sent.items():
        unique = {}
        for e in elist:
            unique.setdefault((e["key"], e["label"]), 0)
            unique[(e["key"], e["label"])] += 1

        keys = list(unique.keys())
        for (k1, t1), (k2, t2) in combinations(keys, 2):
            a, b = sorted([k1, k2])
            edges[(a, b)] += 1
    return edges

In [ ]:
def cooccurrence_tokens(ents, window_size=30):
    ents_sorted = sorted(ents, key=lambda e: e["tok_i"])
    edges = Counter()
    n = len(ents_sorted)
    for i in range(n):
        ei = ents_sorted[i]
        for j in range(i+1, n):
            ej = ents_sorted[j]
            if ej["tok_i"] - ei["tok_i"] > window_size:
                break
            if ei["key"] == ej["key"]:
                continue
            a, b = sorted([ei["key"], ej["key"]])
            edges[(a, b)] += 1
    return edges

In [ ]:
all_ents = []

for doc_name, doc in zip([n for n,_ in texts], nlp.pipe([t for _,t in texts], batch_size=8)):
    all_ents.extend(extract_entities(doc))

key_counts = Counter(e["key"] for e in all_ents)
surface_counts = defaultdict(Counter)
type_counts = defaultdict(Counter)
for e in all_ents:
    surface_counts[e["key"]][e["text"]] += 1
    type_counts[e["key"]][e["label"]] += 1

kept_keys = {k for k,c in key_counts.items() if c >= MIN_ENTITY_COUNT}
ents_filt = [e for e in all_ents if e["key"] in kept_keys]

key_to_label = {k: surface_counts[k].most_common(1)[0][0] for k in kept_keys}
key_to_type  = {k: type_counts[k].most_common(1)[0][0] for k in kept_keys}
key_to_count = {k: int(key_counts[k]) for k in kept_keys}

if COOC_WINDOW == "sentence":
    edge_counts = cooccurrence_sentence(ents_filt)
else:
    edge_counts = cooccurrence_tokens(ents_filt, window_size=WINDOW_SIZE_TOKENS)

edge_counts = Counter({k:w for k,w in edge_counts.items() if w >= MIN_EDGE_WEIGHT})

In [ ]:
nodes_rows = []
for key in sorted(kept_keys):
    ent_type = key_to_type.get(key, "ENT")
    nodes_rows.append({
        "Id": make_id(key, ent_type),
        "Label": key_to_label[key],
        "Type": ent_type,
        "Count": key_to_count[key],
    })

edges_rows = []
for (k1, k2), w in edge_counts.items():
    t1 = key_to_type.get(k1, "ENT")
    t2 = key_to_type.get(k2, "ENT")
    edges_rows.append({
        "Source": make_id(k1, t1),
        "Target": make_id(k2, t2),
        "Type": "Undirected",
        "Weight": int(w),
    })

nodes_df = pd.DataFrame(nodes_rows)
edges_df = pd.DataFrame(edges_rows)

nodes_path = OUTDIR / "nodes.csv"
edges_path = OUTDIR / "edges.csv"
nodes_df.to_csv(nodes_path, index=False)
edges_df.to_csv(edges_path, index=False)

print("Nodes:", len(nodes_df), "Edges:", len(edges_df))
nodes_path, edges_path